# Assignment 11 — Fine-Tuning GPT-2 on a Custom COD Gaming Text Style

**Name:** TEJAS
**Model:** GPT-2 (`gpt2`)  
**Custom style:** Casual Call of Duty / COD Mobile gaming commentary  
**Dataset:** `cod_gaming_dataset.txt`

### Objective
Fine-tune a pretrained GPT-2 model on approximately 200–300 lines of consistent COD gaming text and compare its output with the original GPT-2 model using the same prompt from Assignment 2.

### Text source
The training text is an **original, self-created COD gaming commentary dataset** written for this assignment. It focuses on ranked matches, Battle Royale, loadouts, aiming, movement, positioning, teamwork, communication, rotations and match strategy. No song lyrics or copied article text are used.

**Source file:** `cod_gaming_source.txt`  
**Cleaned training file:** `cod_gaming_dataset.txt`

The source file intentionally contains a few headings, metadata lines and separators. These are removed during cleaning so the final training dataset contains only language text.

In [1]:
!pip install -q transformers datasets accelerate

In [2]:
import re
import os
from pathlib import Path

SOURCE_FILE = "cod_gaming_source.txt"
DATASET_FILE = "cod_gaming_dataset.txt"

print("Source file:", SOURCE_FILE)
print("Dataset file:", DATASET_FILE)

Source file: cod_gaming_source.txt
Dataset file: cod_gaming_dataset.txt


## 1. Load the original COD gaming source text

The source is loaded from the provided TXT file. It contains the original COD gaming commentary plus a small amount of non-language formatting that will be removed during cleaning.

In [5]:
import os
from pathlib import Path

SOURCE_FILE = "/content/cod_gaming_source.txt"

if not os.path.exists(SOURCE_FILE):
    raise FileNotFoundError(
        f"{SOURCE_FILE} was not found. Please upload cod_gaming_source.txt to Colab."
    )

raw_text = Path(SOURCE_FILE).read_text(encoding="utf-8")

print("Source file:", SOURCE_FILE)
print("Raw characters:", len(raw_text))
print("\nFirst 1200 characters:\n")
print(raw_text[:1200])

Source file: /content/cod_gaming_source.txt
Raw characters: 17901

First 1200 characters:

COD GAMING COMMENTARY — ORIGINAL SOURCE TEXT
[Dataset prepared for Assignment 11]
Player: TERrex | Session: Ranked COD Mobile
-----------------------------------------------
The ranked match started with our squad moving quickly toward the first objective.
I checked the minimap before leaving spawn and marked the safest route for my team.
The enemy team pushed the left lane, so I rotated through the middle building.
My teammate called out two opponents near the warehouse and I held the angle.
I picked up my usual assault rifle and prepared for the first gunfight.
The opening fight was close, but our squad secured the position before the enemy arrived.
I heard footsteps behind the wall and waited instead of rushing into the room.
The enemy finally crossed the doorway, and I won the duel with controlled bursts.
Our team captured the objective while I watched the flank from the upper floor.
A sniper

## 2. Clean the text and create approximately 200–300 training lines

The cleaning step removes player metadata, separators, headings, URLs, bullets, extra spaces and other non-language content. The final dataset is checked to keep the target size within the assignment requirement.

In [6]:
def clean_text(text):
    cleaned_lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if re.fullmatch(r"[-_=*~]{5,}", line):
            continue
        if line.startswith("COD GAMING COMMENTARY"):
            continue
        if line.startswith("[Dataset prepared"):
            continue
        if line.startswith("Player:"):
            continue
        if line == "END OF SOURCE":
            continue
        line = re.sub(r"https?://\S+|www\.\S+", "", line)
        line = re.sub(r"^[•*-]+\s*", "", line)
        line = line.replace("\xa0", " ")
        line = re.sub(r"\s+", " ", line)
        line = re.sub(r"\s+([,.;:!?])", r"\1", line)
        line = line.strip()
        if len(line) >= 20:
            cleaned_lines.append(line)

    unique_lines = []
    seen = set()
    for line in cleaned_lines:
        key = line.lower()
        if key not in seen:
            unique_lines.append(line)
            seen.add(key)
    return unique_lines


dataset_lines = clean_text(raw_text)
if len(dataset_lines) < 200:
    raise RuntimeError(f"Only {len(dataset_lines)} usable lines were found. At least 200 are required.")
dataset_lines = dataset_lines[:300]
Path(DATASET_FILE).write_text("\n".join(dataset_lines) + "\n", encoding="utf-8")

print("Generated dataset lines:", len(dataset_lines))
print("Dataset saved to:", DATASET_FILE)
print("\nSample cleaned lines:\n")
for i, line in enumerate(dataset_lines[:10], 1):
    print(f"{i}. {line}")

Generated dataset lines: 240
Dataset saved to: cod_gaming_dataset.txt

Sample cleaned lines:

1. The ranked match started with our squad moving quickly toward the first objective.
2. I checked the minimap before leaving spawn and marked the safest route for my team.
3. The enemy team pushed the left lane, so I rotated through the middle building.
4. My teammate called out two opponents near the warehouse and I held the angle.
5. I picked up my usual assault rifle and prepared for the first gunfight.
6. The opening fight was close, but our squad secured the position before the enemy arrived.
7. I heard footsteps behind the wall and waited instead of rushing into the room.
8. The enemy finally crossed the doorway, and I won the duel with controlled bursts.
9. Our team captured the objective while I watched the flank from the upper floor.
10. A sniper appeared on the hill, so I moved behind cover and changed my position.


## 3. Inspect the cleaned dataset

The cleaned dataset should contain only COD gaming language. Each line is treated as a separate training example.

In [7]:
with open(DATASET_FILE, "r", encoding="utf-8") as f:
    dataset_lines = [line.strip() for line in f if line.strip()]

print("Number of training lines:", len(dataset_lines))
print("Total words:", sum(len(line.split()) for line in dataset_lines))
print("\nFirst 15 training lines:\n")
for i, line in enumerate(dataset_lines[:15], 1):
    print(f"{i}. {line}")

Number of training lines: 240
Total words: 2856

First 15 training lines:

1. The ranked match started with our squad moving quickly toward the first objective.
2. I checked the minimap before leaving spawn and marked the safest route for my team.
3. The enemy team pushed the left lane, so I rotated through the middle building.
4. My teammate called out two opponents near the warehouse and I held the angle.
5. I picked up my usual assault rifle and prepared for the first gunfight.
6. The opening fight was close, but our squad secured the position before the enemy arrived.
7. I heard footsteps behind the wall and waited instead of rushing into the room.
8. The enemy finally crossed the doorway, and I won the duel with controlled bursts.
9. Our team captured the objective while I watched the flank from the upper floor.
10. A sniper appeared on the hill, so I moved behind cover and changed my position.
11. The next rotation was important because the enemy had control of the strongest buil

## 4. Load GPT-2

GPT-2 is loaded as the pretrained base model. The same Assignment 2 prompt is used later for both the original and fine-tuned models.

In [8]:
import torch
from datasets import Dataset
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling

MODEL_NAME = "gpt2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

Device: cpu


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## 5. Prepare the Hugging Face dataset

Each cleaned line is treated as a language-model training example. `DataCollatorForLanguageModeling` prepares labels for causal language-model training.

In [9]:
train_dataset = Dataset.from_dict({"text": dataset_lines})

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(tokenized_dataset)

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 240
})


## 6. Fine-tune GPT-2 with Hugging Face Trainer

The settings below are moderate for Google Colab. A GPU is recommended for faster training.

In [10]:
OUTPUT_DIR = "./gpt2_cod_gaming"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_dataset, data_collator=data_collator)
print("Starting GPT-2 fine-tuning...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("\nFine-tuned model saved to:")
print(OUTPUT_DIR)

Starting GPT-2 fine-tuning...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.339639
20,3.733512
30,3.754578
40,3.088630
50,3.159202
60,3.035012
70,2.761572
80,2.878551
90,2.822656


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuned model saved to:
./gpt2_cod_gaming


## 7. Generate from the original GPT-2 and the fine-tuned GPT-2

This uses the **same Assignment 2 prompt** for both models, as required. The prompt below matches the uploaded reference notebook. Replace it only when your actual Assignment 2 prompt was different.

In [11]:
ASSIGNMENT_2_PROMPT = "Artificial Intelligence is changing"
BASE_MODEL_NAME = "gpt2"

base_tokenizer = GPT2Tokenizer.from_pretrained(BASE_MODEL_NAME)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_model = GPT2LMHeadModel.from_pretrained(BASE_MODEL_NAME).to(device)

ft_tokenizer = GPT2Tokenizer.from_pretrained(OUTPUT_DIR)
ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR).to(device)

def generate_text(model, tokenizer, prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.92,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("Generating base-model output...\n")
base_output = generate_text(base_model, base_tokenizer, ASSIGNMENT_2_PROMPT)
print(base_output)
print("\n" + "=" * 80 + "\n")
print("Generating fine-tuned output...\n")
finetuned_output = generate_text(ft_model, ft_tokenizer, ASSIGNMENT_2_PROMPT)
print(finetuned_output)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generating base-model output...

Artificial Intelligence is changing the way we think about how to improve human productivity, according a new study published in Nature Communications.
 and of which I am one: AI experts will continue studying what makes us great as humans; that would be my work being made available through this project (the data was collected on our PCs); but at least some people are working together with other teams across different fields for projects like Artificial Intelligence & Human Resource Management or HR automation... The researchers also say they're trying not only by building systems capable just-in-time information processing capabilities into their own models so you can see where those machines go


Generating fine-tuned output...

Artificial Intelligence is changing the way we interact with games. Players should consider what makes them more productive and less likely to make mistakes that might lead their team down a different path.
, who can help us i

## 8. Save the outputs for submission

The base-model output and fine-tuned output are saved as separate text files.

In [12]:
with open("base_model_output.txt", "w", encoding="utf-8") as f:
    f.write(base_output)
with open("finetuned_model_output.txt", "w", encoding="utf-8") as f:
    f.write(finetuned_output)
print("Saved:")
print("- base_model_output.txt")
print("- finetuned_model_output.txt")

Saved:
- base_model_output.txt
- finetuned_model_output.txt


## 9. Before-and-after comparison

### Analysis

Compare the actual outputs produced by the two models. At least two clear differences should be identified.

**Difference 1 — Vocabulary:** the fine-tuned model should be more likely to use COD/gaming terms such as match, squad, enemy, loadout, weapon, objective, minimap, rotation or ranked play.

**Difference 2 — Style and topic:** the fine-tuned model should move toward casual gameplay commentary and match-focused sentences instead of the broader continuation produced by base GPT-2.

**Difference 3 — Sentence patterns:** the fine-tuned model may reproduce sentence patterns found in the training data, such as describing enemy encounters, positioning, movement and team communication.

The final written comparison should be based on the actual generated text from the notebook run.

In [13]:
print("BEFORE — Original GPT-2")
print("-" * 80)
print(base_output)
print("\nAFTER — Fine-tuned COD Gaming GPT-2")
print("-" * 80)
print(finetuned_output)
print("\nPrompt used for both models:")
print(ASSIGNMENT_2_PROMPT)

BEFORE — Original GPT-2
--------------------------------------------------------------------------------
Artificial Intelligence is changing the way we think about how to improve human productivity, according a new study published in Nature Communications.
 and of which I am one: AI experts will continue studying what makes us great as humans; that would be my work being made available through this project (the data was collected on our PCs); but at least some people are working together with other teams across different fields for projects like Artificial Intelligence & Human Resource Management or HR automation... The researchers also say they're trying not only by building systems capable just-in-time information processing capabilities into their own models so you can see where those machines go

AFTER — Fine-tuned COD Gaming GPT-2
--------------------------------------------------------------------------------
Artificial Intelligence is changing the way we interact with games. Pla

## 10. Final conclusion

Fine-tuning adapts the pretrained GPT-2 model toward the vocabulary, tone and sentence patterns present in the custom COD gaming dataset. In this experiment, the model is expected to move from a general language style toward COD-focused gameplay commentary. The final conclusion should be supported by the actual before-and-after outputs.